# Table S1 Prompt Tuning Model Performance

Recreates Supplementary Table S1 at the fixed reporting threshold of 0.8 using human labels.


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

REPORTING_THRESHOLD = 0.8

cwd = Path.cwd().resolve()
REPO_ROOT = next(path for path in (cwd, *cwd.parents) if (path / "anonymized_data" / "labelled").exists())
LABELLED_DIR = REPO_ROOT / "anonymized_data" / "labelled"

TABLE_S1_THEORY_ORDER = [
    "5g",
    "antivax",
    "gates",
    "blm",
    "china",
    "democrats",
    "hospitals",
    "fakenews",
    "testing",
    "fauci",
    "hydroxy",
    "deaths",
    "pizzagate",
    "plandemic",
    "qanon",
    "trumppuppet",
]

TABLE_S1_LABELS = {
    "5g": "5G",
    "antivax": "Anti-Vaccine",
    "gates": "Bill Gates",
    "blm": "Black Lives Matters",
    "china": "China Created Covid",
    "democrats": "Democrats",
    "hospitals": "Empty Hospitals",
    "fakenews": "Fake News",
    "testing": "False Positive Testing",
    "fauci": "Anthony Fauci",
    "hydroxy": "Hydroxychloroquine",
    "deaths": "Miscounted Deaths",
    "pizzagate": "Pizzagate",
    "plandemic": "Plandemic",
    "qanon": "Qanon",
    "trumppuppet": "Trump Puppet",
}


def score_metrics(scores, labels, threshold):
    scores = np.asarray(scores, dtype=float)
    labels = np.asarray(labels, dtype=int)
    predictions = scores >= threshold

    positives = labels == 1
    negatives = labels == 0

    tp = int(np.sum(predictions & positives))
    fp = int(np.sum(predictions & negatives))
    tn = int(np.sum((~predictions) & negatives))
    fn = int(np.sum((~predictions) & positives))

    precision = tp / (tp + fp) if (tp + fp) else 0.0
    recall = tp / (tp + fn) if (tp + fn) else 0.0
    accuracy = (tp + tn) / (tp + fp + tn + fn) if (tp + fp + tn + fn) else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0

    return {"accuracy": accuracy, "precision": precision, "recall": recall, "f1": f1}


table_s1_rows = []

for theory in TABLE_S1_THEORY_ORDER:
    csv_path = LABELLED_DIR / f"labelled_{theory}.csv"
    raw = pd.read_csv(csv_path, low_memory=False)
    raw.columns = raw.columns.str.strip()
    raw["text_label_clean"] = raw["text_label"].astype("string").str.strip().str.lower()
    raw["yes_prob"] = pd.to_numeric(raw["yes_prob"], errors="coerce")

    is_human = raw["text_label_clean"].isin(["yes", "no"])
    validation = raw.loc[is_human, ["text_label_clean", "yes_prob"]].dropna(subset=["yes_prob"])
    y_true = (validation["text_label_clean"] == "yes").astype(int)
    metrics = score_metrics(validation["yes_prob"], y_true, REPORTING_THRESHOLD)

    table_s1_rows.append({
        "Conspiracy": TABLE_S1_LABELS[theory],
        "Keyword Match Tweet Count": len(raw),
        "Human Label Rows": len(validation),
        "Accuracy": metrics["accuracy"],
        "Precision": metrics["precision"],
        "Recall": metrics["recall"],
        "F1": metrics["f1"],
    })

table_s1 = pd.DataFrame(table_s1_rows)
table_s1_display = table_s1.copy()
table_s1_display["Keyword Match Tweet Count"] = table_s1_display[
    "Keyword Match Tweet Count"
].map("{:,}".format)
table_s1_display["Human Label Rows"] = table_s1_display["Human Label Rows"].map("{:,}".format)
table_s1_display["Accuracy"] = table_s1_display["Accuracy"].map("{:.2f}".format)
table_s1_display["Precision"] = table_s1_display["Precision"].map("{:.2f}".format)
table_s1_display["Recall"] = table_s1_display["Recall"].map("{:.2f}".format)
table_s1_display["F1"] = table_s1_display["F1"].map("{:.2f}".format)

latex_caption = (
    "Prompt tuning model performance evaluated at the fixed endorsement "
    "probability threshold of 0.8, chosen to reduce false positives. "
    "Keyword Match Tweet Count is the number "
    "of keyword matched candidate tweets in each labelled CSV. Human Label "
    "Rows is the subset with human yes or no labels and numeric model "
    "probabilities. Accuracy, precision, recall, and F1 compare the thresholded model "
    "predictions against those human labels."
)

display(table_s1_display)
print(table_s1_display.to_latex(
    index=False,
    escape=True,
    caption=latex_caption,
    label="tab:s1_prompt_tuning_model_performance",
))
